# 股價情緒分析 Pipeline（整合版）

Big Data & Business Analytics, National Taiwan University

## Pipeline 架構

| Step | 功能 | 主要輸出 |
|------|------|---------|
| 0 | 總參數區與共用工具 | `DB_CONFIG`, utils |
| 1 | 從 `stock_text` 篩選公司相關文章並去噪 | `articles_raw.csv` |
| 2 | 依 D+n 交易日漲跌幅貼上 1 / -1 / 0 標籤 | `articles_labeled.csv` |
| 3 | 斷詞、詞彙篩選、TF-IDF 向量化 | `articles_processed.csv`, `limit_model.npz` |
| 4 | 訓練 5 種分類器並評估 | 混淆矩陣、準確率 |
| 5 | 移動回測（真實預測情境） | 逐日預測明細、回測混淆矩陣 |

## 方法切換參數（method switches）

| 參數 | 選項 | 說明 |
|------|------|------|
| `TERM_SELECT_METHOD` | `'tf'` / `'chi2'` / `'sklearn'` | Step 3 詞彙選取方法 |
| `TOKENIZE_METHOD` | `'monpa'` / `'ngram'` | Step 3 / Step 5 斷詞方法 |
| `SPLIT_METHOD` | `'time'` / `'random'` | Step 4 訓練/測試切割方式 |

---
# Step 0 — 總參數區與共用工具

所有可調參數集中於此；各 Step 開頭若有覆寫需求，可在該 Step 的參數覆寫格中重新指派。

In [21]:
# ═══════════════ 總參數區 ═══════════════

# ── 目標公司 ──
COMPANY_ID   = '1519'
COMPANY_NAME = '華城'

# ── Step 1：篩選關鍵字 ──
INCLUDE_KEYWORDS = [
    '華城', '華城電機', '華城電能', 'EVALUE',
    'Fortune Electric', '重電三雄', '重電四君子'
]
EXCLUDE_KEYWORDS = [
    '京華城', '柯文哲', '橘子', '許芷瑜',
    '沈慶京', '應曉薇', '容積率', '弊案', '羈押'
]

# ── Step 2：貼標參數 ──
N_DAYS = 3       # D+n 交易日後的漲跌
SIGMA  = 0.03    # 漲跌門檻（3%）

# ── Step 3：斷詞與向量化 ──
TOKENIZE_METHOD    = 'monpa'   # 'monpa' | 'ngram'
TERM_SELECT_METHOD = 'sklearn' # 'tf' | 'chi2' | 'sklearn'
TOP_K_WORDS        = 200       # 看漲/看跌各取 K 個（sklearn 模式則為 K*2 總特徵數）
NGRAM_RANGE        = (1, 3)    # 詞級 N-gram 範圍（1=unigram, 2=bigram, 3=trigram）
                               # 先以 monpa 斷成詞，再由 TF-IDF 做詞組合
NGRAM_MIN          = 2         # TOKENIZE_METHOD='ngram' 時使用的字元 N-gram 最小長度
NGRAM_MAX          = 4         # TOKENIZE_METHOD='ngram' 時使用的字元 N-gram 最大長度

# ── Step 4：分類器訓練 ──
SPLIT_METHOD = 'time'    # 'time'（依時序切）| 'random'（stratified 隨機切）
TEST_RATIO   = 0.2
RANDOM_SEED  = 42

# ── Step 5：移動回測 ──
WINDOW_DAYS  = 30    # 訓練窗口（日曆天）
MIN_ARTICLES = 3     # 當日文章篇數門檻（低於此不出手）

# ── 檔名輸出 ──
PATH_ARTICLES_RAW       = 'articles_raw.csv'
PATH_ARTICLES_FILTERED  = 'articles_filtered_out.csv'
PATH_ARTICLES_LABELED   = 'articles_labeled.csv'
PATH_ARTICLES_PROCESSED = 'articles_processed.csv'
PATH_VECTOR_NPZ         = 'limit_model.npz'
PATH_FEATURE_NAMES      = 'feature_names.pkl'
PATH_BULL_TERMS         = f'top_{TOP_K_WORDS}_bull_terms.csv'
PATH_BEAR_TERMS         = f'top_{TOP_K_WORDS}_bear_terms.csv'

In [22]:
# ── 資料庫連線設定（讀 .env）──
import os
from dotenv import load_dotenv

load_dotenv()

DB_CONFIG = {
    'host':     os.getenv('DB_HOST',     'localhost'),
    'user':     os.getenv('DB_USER',     'root'),
    'password': os.getenv('DB_PASSWORD', ''),
    'database': os.getenv('DB_NAME',     'bda2026'),
    'charset':  os.getenv('DB_CHARSET',  'utf8mb4'),
}

In [23]:
# ── 共用工具函式 ──
import re
import bisect
import csv
from collections import Counter, defaultdict
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import pymysql
from scipy import sparse


def db_connect():
    """建立 MySQL 連線。"""
    return pymysql.connect(**DB_CONFIG)


def is_valid_token(token: str) -> bool:
    """過濾 URL、純英數、雜訊 token。中文字需佔 80% 以上。"""
    if not token or len(token) < 2:
        return False
    if re.search(r'https?://', token):
        return False
    if re.search(r'www\.', token):
        return False
    chinese_chars = len(re.findall(r'[\u4e00-\u9fff]', token))
    if chinese_chars == 0:
        return False
    if chinese_chars / len(token) < 0.8:
        return False
    return True


def get_ngrams(text: str, min_n: int = NGRAM_MIN, max_n: int = NGRAM_MAX) -> list:
    """N-gram 字元切割：只保留中/英/數，去空白與純標點。"""
    clean_text = ''.join(re.findall(r'[\u4e00-\u9fffA-Za-z0-9]+', text))
    ngrams = []
    for n in range(min_n, max_n + 1):
        for i in range(len(clean_text) - n + 1):
            gram = clean_text[i:i + n]
            if re.search(r'[\u4e00-\u9fffA-Za-z]', gram):
                ngrams.append(gram)
    return ngrams


def tokenize(text: str, method: str = None) -> list:
    """依 TOKENIZE_METHOD 切割文字並過濾雜訊 token。"""
    method = method or TOKENIZE_METHOD

    if method == 'monpa':
        import monpa
        from monpa import utils as monpa_utils
        tokens = []
        for sent in monpa_utils.short_sentence(text):
            for term in monpa.cut(sent):
                term = term.strip()
                if len(term) > 1:
                    tokens.append(term)
    elif method == 'ngram':
        tokens = get_ngrams(text)
    else:
        raise ValueError(f'未知的 TOKENIZE_METHOD: {method}')

    return [t for t in tokens if is_valid_token(t)]


print('共用工具已載入')

共用工具已載入


---
# Step 1 — 篩選相關文章

從 `stock_text` 以公司關鍵字擷取主文，去除雜訊（京華城弊案）、排行榜、數字比例過高的文章。

In [24]:
# ── Step 1 參數覆寫（如需）──
# 例：INCLUDE_KEYWORDS = [...]

# 取得股價資料的時間範圍（作為文章篩選的時間邊界）
conn = db_connect()
cursor = conn.cursor()
cursor.execute(
    'SELECT MIN(trade_date), MAX(trade_date) FROM stock_prices WHERE company_id = %s',
    (COMPANY_ID,)
)
min_date, max_date = cursor.fetchone()
print(f'股價資料範圍：{min_date} ～ {max_date}')

# 動態組合 SQL：包含條件 + 排除條件 + 時間範圍
search_keywords = INCLUDE_KEYWORDS + [COMPANY_ID]

include_clause = ' OR '.join(['(title LIKE %s OR content LIKE %s)'] * len(search_keywords))
include_params = [p for kw in search_keywords for p in (f'%{kw}%', f'%{kw}%')]

if EXCLUDE_KEYWORDS:
    exclude_clause = ' AND ' + ' AND '.join(
        ['(title NOT LIKE %s AND content NOT LIKE %s)'] * len(EXCLUDE_KEYWORDS)
    )
    exclude_params = [p for kw in EXCLUDE_KEYWORDS for p in (f'%{kw}%', f'%{kw}%')]
else:
    exclude_clause, exclude_params = '', []

sql_query = f"""
    SELECT no, post_time, title, content, s_name
    FROM stock_text
    WHERE ({include_clause})
      {exclude_clause}
      AND (content_type = 'main' OR content_type IS NULL)
      AND DATE(post_time) >= %s
      AND DATE(post_time) <= %s
    ORDER BY post_time
"""

final_params = include_params + exclude_params + [min_date, max_date]

print('正在進行深度篩選與去噪...')
cursor.execute(sql_query, final_params)
rows = cursor.fetchall()
print(f'關鍵字精確篩選後：{len(rows)} 篇')

cursor.close()
conn.close()

股價資料範圍：2023-03-01 ～ 2025-02-27
正在進行深度篩選與去噪...
關鍵字精確篩選後：4105 篇


In [25]:
# ── 去除 boilerplate：排行榜、數字比例過高、太短的文章 ──
BOILERPLATE_TITLES = ['買賣超排行', '投信買賣', '外資買賣', '主力買賣', '法人買賣']


def is_boilerplate(title: str, content: str) -> bool:
    text = (title or '') + (content or '')
    if any(p in (title or '') for p in BOILERPLATE_TITLES):
        return True
    if len(text) == 0:
        return True
    if sum(c.isdigit() for c in text) / len(text) > 0.3:
        return True
    if len(text) < 50:
        return True
    return False


# rows 欄位順序：no, post_time, title, content, s_name
rows_kept    = [r for r in rows if not is_boilerplate(r[2], r[3])]
rows_removed = [r for r in rows if     is_boilerplate(r[2], r[3])]

print(f'篩選前：{len(rows)} 篇')
print(f'過濾掉：{len(rows_removed)} 篇')
print(f'保留：  {len(rows_kept)} 篇')

篩選前：4105 篇
過濾掉：800 篇
保留：  3305 篇


In [26]:
# ── 儲存 CSV ──
HEADER = ['no', 'post_time', 'title', 'content', 's_name']

for path, data, label in [
    (PATH_ARTICLES_RAW,      rows_kept,    '保留'),
    (PATH_ARTICLES_FILTERED, rows_removed, '過濾'),
]:
    with open(path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(HEADER)
        writer.writerows(data)
    print(f'已儲存 {path}（{label}）：{len(data)} 筆')

# 各平台分布統計
for data, label in [(rows_kept, '保留'), (rows_removed, '過濾')]:
    print(f'\n===== {label}文章 - 各平台分布（{len(data)} 筆）=====')
    for src, cnt in Counter(r[4] for r in data).most_common():
        print(f'  {src}: {cnt} 篇')

已儲存 articles_raw.csv（保留）：3305 筆
已儲存 articles_filtered_out.csv（過濾）：800 筆

===== 保留文章 - 各平台分布（3305 筆）=====
  Yahoo股市: 2363 篇
  Yahoo新聞: 347 篇
  Ptt: 335 篇
  Dcard: 255 篇
  Mobile01: 5 篇

===== 過濾文章 - 各平台分布（800 筆）=====
  Ptt: 523 篇
  Yahoo股市: 222 篇
  Dcard: 29 篇
  Yahoo新聞: 26 篇


---
# Step 2 — 貼股價標籤

對每篇文章，取其發文日（或下一個交易日）的收盤價，與 D+n 交易日後收盤價比較：
- 漲幅 > σ → **+1（看漲）**
- 跌幅 < -σ → **-1（看跌）**
- 否則 → **0（中性，不納入訓練）**

In [27]:
# ── Step 2 參數覆寫 ──
# 例：N_DAYS = 5; SIGMA = 0.02

# 讀該公司所有交易日與收盤價
conn = db_connect()
cursor = conn.cursor()
cursor.execute(
    '''
    SELECT trade_date, closing_price
    FROM stock_prices
    WHERE company_id = %s
    ORDER BY trade_date
    ''',
    (COMPANY_ID,)
)
price_rows = cursor.fetchall()
cursor.close()
conn.close()

trade_dates = [row[0] for row in price_rows]
price_dict  = {row[0]: float(row[1]) for row in price_rows}

print(f'共 {len(trade_dates)} 個交易日')
print(f'範圍：{trade_dates[0]} ～ {trade_dates[-1]}')

共 483 個交易日
範圍：2023-03-01 ～ 2025-02-27


In [28]:
# ── 交易日查詢輔助函式 ──
# [METHOD] 目前方法：固定 n 個交易日後 | 備用：事件窗口法（取最大漲跌）

def get_price_on_or_after(date, trade_dates, price_dict):
    """date 當天或之後第一個交易日的收盤價"""
    idx = bisect.bisect_left(trade_dates, date)
    if idx >= len(trade_dates):
        return None
    return price_dict[trade_dates[idx]]


def get_nth_trading_day_price(date, n, trade_dates, price_dict):
    """date 之後第 n 個交易日（不含當日）的收盤價"""
    idx = bisect.bisect_right(trade_dates, date)
    target_idx = idx + n - 1
    if target_idx >= len(trade_dates):
        return None
    return price_dict[trade_dates[target_idx]]

In [29]:
# ── 對每篇文章貼標籤 ──
articles = []
with open(PATH_ARTICLES_RAW, newline='', encoding='utf-8') as f:
    articles = list(csv.DictReader(f))
print(f'讀入 {len(articles)} 篇文章')

labeled = []
skip_count = 0

for art in articles:
    try:
        post_date = datetime.strptime(art['post_time'], '%Y-%m-%d %H:%M:%S').date()
    except Exception:
        skip_count += 1
        continue

    price_d  = get_price_on_or_after(post_date, trade_dates, price_dict)
    price_dn = get_nth_trading_day_price(post_date, N_DAYS, trade_dates, price_dict)

    if price_d is None or price_dn is None or price_d == 0:
        skip_count += 1
        continue

    pct = (price_dn - price_d) / price_d

    if pct > SIGMA:
        label = 1
    elif pct < -SIGMA:
        label = -1
    else:
        label = 0

    art['label']      = label
    art['pct_change'] = round(pct * 100, 2)
    labeled.append(art)

print(f'貼標完成：{len(labeled)} 篇（跳過 {skip_count} 篇）')

# 儲存
fieldnames = ['no', 'post_time', 'title', 'content', 's_name', 'label', 'pct_change']
with open(PATH_ARTICLES_LABELED, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for art in labeled:
        writer.writerow({k: art.get(k, '') for k in fieldnames})

# 標籤分布
label_cnt = Counter(art['label'] for art in labeled)
print(f'看漲 ( 1)：{label_cnt[ 1]} 篇')
print(f'看跌 (-1)：{label_cnt[-1]} 篇')
print(f'中性 ( 0)：{label_cnt[ 0]} 篇（不納入訓練）')
print(f'已儲存至 {PATH_ARTICLES_LABELED}')

讀入 3305 篇文章
貼標完成：3296 篇（跳過 9 篇）
看漲 ( 1)：1186 篇
看跌 (-1)：821 篇
中性 ( 0)：1289 篇（不納入訓練）
已儲存至 articles_labeled.csv


---
# Step 3 — 斷詞、詞彙篩選、向量化

## 方法切換

- **`TOKENIZE_METHOD`**：
  - `'monpa'`：繁中分詞器（較慢，語意較精準）
  - `'ngram'`：字元 N-gram（快，無語言知識依賴）

- **`TERM_SELECT_METHOD`**：
  - `'tf'`：依詞頻排序，各取 TOP_K 個看漲/看跌鑑別詞，輸出 `bull_terms.csv` / `bear_terms.csv`
  - `'chi2'`：依卡方值排序（手動計算 bull/bear DF 偏向），輸出同上
  - `'sklearn'`：直接用 sklearn 的 `SelectKBest(chi2)` 篩出 TOP_K*2 個總特徵

In [ ]:
# ── Step 3 參數覆寫 ──
# 例：TOKENIZE_METHOD = 'ngram'; TERM_SELECT_METHOD = 'chi2'

# 讀入已貼標文章，排除中性（label=0）
articles = []
with open(PATH_ARTICLES_LABELED, newline='', encoding='utf-8') as f:
    for row in csv.DictReader(f):
        row['label'] = int(row['label'])
        articles.append(row)

bull_arts = [a for a in articles if a['label'] ==  1]
bear_arts = [a for a in articles if a['label'] == -1]
print(f'看漲文章：{len(bull_arts)} 篇')
print(f'看跌文章：{len(bear_arts)} 篇')

# ── 逐篇斷詞（保留 post_date 供後續每天合併）──
print(f'\n斷詞中（method={TOKENIZE_METHOD}）...')
_corpus_raw   = []  # 每篇的 tokens_str
_labels_raw   = []  # 每篇 label
_dates_raw    = []  # 每篇 post_date

for i, art in enumerate(articles):
    if i % 200 == 0:
        print(f'  {i}/{len(articles)}')
    if art['label'] == 0:
        continue

    post_date = datetime.strptime(art['post_time'], '%Y-%m-%d %H:%M:%S').date()
    text = (art.get('title') or '') + ' ' + (art.get('content') or '')
    tokens = tokenize(text)
    _corpus_raw.append(' '.join(tokens))
    _labels_raw.append(art['label'])
    _dates_raw.append(post_date)

print(f'\n斷詞完成，共 {len(_corpus_raw)} 篇（排除中性）')

# ── 每天合併成一筆（與 stock_v2 相同：同日所有文章串接為一份文件）──
# 同一天的所有文章合併後，以 label = 該日第一篇非中性文章的標籤為準
# （同日所有非中性文章標籤相同，因為標籤來自同一天股價）
_day_tokens  = {}   # date -> [tokens_str, ...]
_day_label   = {}   # date -> label

for d, t, l in zip(_dates_raw, _corpus_raw, _labels_raw):
    if d not in _day_tokens:
        _day_tokens[d]  = []
        _day_label[d]   = l
    _day_tokens[d].append(t)

daily_dates = sorted(_day_tokens.keys())
corpus  = [' '.join(_day_tokens[d]) for d in daily_dates]   # 每天一筆
labels  = [_day_label[d]            for d in daily_dates]

print(f'每天合併後：{len(corpus)} 天（原 {len(_corpus_raw)} 篇文章）')
print(f'看漲天數：{labels.count(1)}，看跌天數：{labels.count(-1)}')

# 儲存（每天一列，含 date 欄位供 Step 5 參考）
with open(PATH_ARTICLES_PROCESSED, 'w', newline='', encoding='utf-8-sig') as f:
    writer = csv.writer(f)
    writer.writerow(['date', 'processed_text', 'label'])
    for d, text, label in zip(daily_dates, corpus, labels):
        writer.writerow([d, text, label])
print(f'已儲存至 {PATH_ARTICLES_PROCESSED}')

In [ ]:
# ── 詞彙篩選：依 TERM_SELECT_METHOD 分流 ──
# 注意：TF-IDF 向量器統一加入 NGRAM_RANGE，
# 讓 monpa 斷出的「詞」再組成 bigram / trigram 詞組合（如「外資 買超」）

def compute_tf_df(corpus_list):
    """計算 TF（總詞頻）與 DF（文件頻率）。僅用於 'tf' 模式（unigram）。"""
    tf, df = Counter(), Counter()
    for text in corpus_list:
        grams = text.split()
        tf.update(grams)
        df.update(set(grams))
    return tf, df


def save_terms_csv(path, words, extra_cols_fn, header):
    with open(path, 'w', newline='', encoding='utf-8-sig') as f:
        writer = csv.writer(f)
        writer.writerow(header)
        for w in words:
            writer.writerow([w] + extra_cols_fn(w))


bull_selected, bear_selected = None, None

if TERM_SELECT_METHOD in ('tf', 'chi2'):
    bull_corpus = [c for c, l in zip(corpus, labels) if l ==  1]
    bear_corpus = [c for c, l in zip(corpus, labels) if l == -1]
    bull_tf, bull_df = compute_tf_df(bull_corpus)
    bear_tf, bear_df = compute_tf_df(bear_corpus)

    if TERM_SELECT_METHOD == 'tf':
        # [METHOD] 依 TF 詞頻排序（unigram 層級）
        # 注意：向量化時仍會套用 NGRAM_RANGE，但 vocabulary 只含 unigram
        bull_selected = [w for w, _ in bull_tf.most_common(TOP_K_WORDS)]
        bear_selected = [w for w, _ in bear_tf.most_common(TOP_K_WORDS)]

        save_terms_csv(PATH_BULL_TERMS, bull_selected,
                       lambda w: [bull_tf[w], bull_df[w]], ['Word', 'TF', 'DF'])
        save_terms_csv(PATH_BEAR_TERMS, bear_selected,
                       lambda w: [bear_tf[w], bear_df[w]], ['Word', 'TF', 'DF'])
        print(f'已依 TF 排序輸出 {PATH_BULL_TERMS} / {PATH_BEAR_TERMS}')

    elif TERM_SELECT_METHOD == 'chi2':
        # [METHOD] 依 sklearn chi2 + bull/bear DF 偏向拆分
        # 用 NGRAM_RANGE 建 TF 矩陣，chi2 分數同時涵蓋詞與詞組合
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.feature_selection import chi2

        vec_full = TfidfVectorizer(use_idf=False, lowercase=False,
                                   token_pattern=r'\S+', ngram_range=NGRAM_RANGE)
        X_full   = vec_full.fit_transform(corpus)
        vocab    = vec_full.get_feature_names_out()

        y_binary = np.array([1 if l == 1 else 0 for l in labels])
        chi2_scores, chi2_pval = chi2(X_full, y_binary)
        word_chi2 = dict(zip(vocab, chi2_scores))
        word_pval = dict(zip(vocab, chi2_pval))

        # 以 unigram DF 判斷主屬類別；N-gram 詞組依 chi2 值排序
        bull_selected = sorted(
            [w for w in vocab if bull_df.get(w.split()[0], 0) >  bear_df.get(w.split()[0], 0)],
            key=lambda w: word_chi2[w], reverse=True
        )[:TOP_K_WORDS]
        bear_selected = sorted(
            [w for w in vocab if bear_df.get(w.split()[0], 0) >= bull_df.get(w.split()[0], 0)],
            key=lambda w: word_chi2[w], reverse=True
        )[:TOP_K_WORDS]

        for path, words in [(PATH_BULL_TERMS, bull_selected),
                             (PATH_BEAR_TERMS, bear_selected)]:
            is_bull = (path == PATH_BULL_TERMS)
            tf_dict = bull_tf if is_bull else bear_tf
            df_dict = bull_df if is_bull else bear_df
            save_terms_csv(
                path, words,
                lambda w: [round(word_chi2[w], 4), round(word_pval[w], 6),
                           tf_dict.get(w.split()[0], 0), df_dict.get(w.split()[0], 0)],
                ['Word', 'Chi2', 'P-value', 'TF(head)', 'DF(head)'],
            )
        print(f'已依 Chi2 排序輸出 {PATH_BULL_TERMS} / {PATH_BEAR_TERMS}')

    print(f'看漲詞：{len(bull_selected)} 個，看跌詞：{len(bear_selected)} 個')

else:
    print(f'TERM_SELECT_METHOD={TERM_SELECT_METHOD}，詞表由 sklearn SelectKBest 於下一格處理')

In [ ]:
# ── 向量化（統一套用 NGRAM_RANGE：詞級 N-gram）──
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2

if TERM_SELECT_METHOD in ('tf', 'chi2'):
    # 以 bull_selected ∪ bear_selected 作為固定 vocabulary
    termset = {}
    for w in list(bull_selected) + list(bear_selected):
        if w not in termset:
            termset[w] = len(termset)

    vectorizer = TfidfVectorizer(vocabulary=termset, use_idf=True, lowercase=False,
                                 token_pattern=r'\S+', ngram_range=NGRAM_RANGE)
    X = vectorizer.fit_transform(corpus)
    feature_names = np.array(list(termset.keys()))
    print(f'方法={TERM_SELECT_METHOD}：vocabulary 固定為 {X.shape[1]} 個詞（含 N-gram 詞組）')

elif TERM_SELECT_METHOD == 'sklearn':
    # 全詞 TF-IDF（含 NGRAM_RANGE）→ SelectKBest(chi2) 篩 TOP_K * 2 個特徵
    vectorizer = TfidfVectorizer(use_idf=True, lowercase=False,
                                 token_pattern=r'\S+', ngram_range=NGRAM_RANGE)
    X_full = vectorizer.fit_transform(corpus)
    print(f'原始矩陣：{X_full.shape}（含 {NGRAM_RANGE[0]}~{NGRAM_RANGE[1]}-gram 詞組）')

    y = np.array(labels)
    selector = SelectKBest(chi2, k=TOP_K_WORDS * 2)
    X = selector.fit_transform(X_full, y)
    feature_names = vectorizer.get_feature_names_out()[selector.get_support()]
    print(f'選後矩陣：{X.shape}')
    print(f'選出特徵（前 20）：{feature_names[:20].tolist()}')

else:
    raise ValueError(f'未知的 TERM_SELECT_METHOD: {TERM_SELECT_METHOD}')

# 儲存向量矩陣與特徵名稱
sparse.save_npz(PATH_VECTOR_NPZ, X)
import pickle
with open(PATH_FEATURE_NAMES, 'wb') as f:
    pickle.dump(list(feature_names), f)

print(f'\n向量矩陣已儲存至 {PATH_VECTOR_NPZ}')
print(f'形狀：{X.shape}（天數 × 特徵數）')
print(f'特徵詞彙已儲存至 {PATH_FEATURE_NAMES}')

In [14]:
# ── 預覽向量矩陣（前 5 列 × 前 10 欄）──
df_preview = pd.DataFrame(
    X[:5, :10].toarray(),
    columns=feature_names[:10],
)
df_preview

,一事,一切,一詮,上來,上千,上游,上盤,上調,上限,下車
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


---
# Step 4 — 分類器訓練與評估

## 切割方式切換（`SPLIT_METHOD`）

- `'time'`：依時序切（前 80% 訓練 / 後 20% 測試），**符合金融場景，避免未來資料洩漏**
- `'random'`：`train_test_split` + `stratify`，標籤分布均衡但會洩漏時序資訊

## 分類器

NB / KNN / SVM / Decision Tree / Random Forest

In [ ]:
# ── Step 4 參數覆寫 ──
# 例：SPLIT_METHOD = 'time'

from sklearn.model_selection import train_test_split

# 載入向量矩陣
X = sparse.load_npz(PATH_VECTOR_NPZ)

# 載入標籤（articles_processed.csv 新增了 date 欄位，y 只需讀 label）
y = []
with open(PATH_ARTICLES_PROCESSED, newline='', encoding='utf-8-sig') as f:
    for row in csv.DictReader(f):
        y.append(int(row['label']))
y = np.array(y)

print(f'共 {X.shape[0]} 天，{X.shape[1]} 維特徵')
print(f'看漲：{sum(y == 1)} 天，看跌：{sum(y == -1)} 天')

# 切割
if SPLIT_METHOD == 'time':
    # [METHOD] 依時序切（corpus 已依 post_date 排序）
    split_idx = int(X.shape[0] * (1 - TEST_RATIO))
    X_train, X_test = X[:split_idx], X[split_idx:]
    y_train, y_test = y[:split_idx], y[split_idx:]
    print(f'\n切割方式：時序（前 {int((1-TEST_RATIO)*100)}% / 後 {int(TEST_RATIO*100)}%）')

elif SPLIT_METHOD == 'random':
    # [METHOD] 隨機 stratified 切
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_RATIO, random_state=RANDOM_SEED, stratify=y
    )
    print(f'\n切割方式：隨機 stratified（test_size={TEST_RATIO}, seed={RANDOM_SEED}）')

else:
    raise ValueError(f'未知的 SPLIT_METHOD: {SPLIT_METHOD}')

print(f'訓練集：{X_train.shape[0]} 天（漲={sum(y_train== 1)}, 跌={sum(y_train==-1)}）')
print(f'測試集：{X_test.shape[0]} 天（漲={sum(y_test == 1)}, 跌={sum(y_test ==-1)}）')

In [16]:
# ── 評估函式 ──
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors   import KNeighborsClassifier
from sklearn.svm         import SVC
from sklearn.tree        import DecisionTreeClassifier
from sklearn.ensemble    import RandomForestClassifier
from sklearn.metrics     import accuracy_score, confusion_matrix, classification_report


def evaluate(name, model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    acc = accuracy_score(y_te, y_pred)
    cm  = confusion_matrix(y_te, y_pred, labels=[1, -1])

    print(f'\n===== {name} =====')
    print(f'準確率：{acc * 100:.1f}%\n')
    print('Classification Report:')
    print(classification_report(y_te, y_pred, labels=[1, -1],
                                target_names=['看漲', '看跌']))
    print('Confusion Matrix（列=真實，欄=預測）：')
    print(f'{"":10s}預測漲  預測跌')
    print(f'真實漲    {cm[0][0]:5d}  {cm[0][1]:5d}')
    print(f'真實跌    {cm[1][0]:5d}  {cm[1][1]:5d}')
    return acc

In [17]:
# ── 訓練 5 種分類器 ──
models = [
    ('Naive Bayes',   MultinomialNB()),
    ('KNN (k=5)',     KNeighborsClassifier(n_neighbors=5)),
    ('SVM',           SVC(kernel='linear')),
    ('Decision Tree', DecisionTreeClassifier(random_state=RANDOM_SEED)),
    ('Random Forest', RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED)),
]

results = []
for name, model in models:
    acc = evaluate(name, model, X_train, y_train, X_test, y_test)
    results.append((name, acc))

print('\n' + '=' * 40)
print('準確率彙整')
print('=' * 40)
for name, acc in sorted(results, key=lambda x: x[1], reverse=True):
    print(f'  {name:20s}: {acc * 100:.1f}%')


===== Naive Bayes =====
準確率：62.2%

Classification Report:
              precision    recall  f1-score   support

          看漲       0.61      1.00      0.76       238
          看跌       1.00      0.07      0.14       164

    accuracy                           0.62       402
   macro avg       0.81      0.54      0.45       402
weighted avg       0.77      0.62      0.50       402

Confusion Matrix（列=真實，欄=預測）：
          預測漲  預測跌
真實漲      238      0
真實跌      152     12

===== KNN (k=5) =====
準確率：68.2%

Classification Report:
              precision    recall  f1-score   support

          看漲       0.68      0.87      0.76       238
          看跌       0.68      0.41      0.52       164

    accuracy                           0.68       402
   macro avg       0.68      0.64      0.64       402
weighted avg       0.68      0.68      0.66       402

Confusion Matrix（列=真實，欄=預測）：
          預測漲  預測跌
真實漲      206     32
真實跌       96     68

===== SVM =====
準確率：60.7%

Classification Report:
   

---
# Step 5 — 移動回測

模擬真實預測場景：逐日往後推，每日以**前 `WINDOW_DAYS` 個日曆天**的文章訓練，預測當日文章多數方向。

## 三層出手門檻

1. 當日有效文章（排除中性）≥ `MIN_ARTICLES`
2. 訓練集須同時包含看漲與看跌樣本
3. 預測結果的多空差距 / 總數 ≥ `SIGNAL_RATIO`（信號要夠強）

## 向量化方式

為與 Step 3 保持一致，Step 5 使用**相同的 `feature_names` 與 `TOKENIZE_METHOD`**重新組裝訓練/測試向量。

In [ ]:
# ── Step 5 參數覆寫 ──
# 例：WINDOW_DAYS = 60

import pickle

# 載入 feature_names（若 Step 3 已執行則直接取用，否則從 pkl 讀入）
if 'feature_names' not in vars():
    with open(PATH_FEATURE_NAMES, 'rb') as f:
        feature_names = np.array(pickle.load(f))
    print(f'從 {PATH_FEATURE_NAMES} 載入 feature_names（{len(feature_names)} 個）')

# 讀入已貼標文章
articles = []
with open(PATH_ARTICLES_LABELED, newline='', encoding='utf-8') as f:
    for row in csv.DictReader(f):
        row['label'] = int(row['label'])
        try:
            row['post_date'] = datetime.strptime(row['post_time'], '%Y-%m-%d %H:%M:%S').date()
            articles.append(row)
        except Exception:
            pass

print(f'讀入 {len(articles)} 篇文章')

# ── 預先建立每天的合併文本（tokenize 一次，避免回測迴圈重複斷詞）──
by_date = defaultdict(list)
for art in articles:
    by_date[art['post_date']].append(art)

all_dates = sorted(by_date.keys())
print(f'共 {len(all_dates)} 個日期')
print(f'預先斷詞中（共 {len(articles)} 篇）...')

day_texts  = {}   # date -> 當日所有文章合併後的 tokens_str
day_labels = {}   # date -> label（只記錄非中性日）
day_counts = {}   # date -> 文章篇數

for d in all_dates:
    arts = by_date[d]
    day_counts[d] = len(arts)
    # 合併當日所有文章的 token
    all_tokens = []
    for art in arts:
        text = (art.get('title') or '') + ' ' + (art.get('content') or '')
        all_tokens.extend(tokenize(text))
    day_texts[d] = ' '.join(all_tokens)
    # 取非中性標籤
    non_neutral = [a for a in arts if a['label'] != 0]
    if non_neutral:
        day_labels[d] = non_neutral[0]['label']

labeled_dates = [d for d in all_dates if d in day_labels]
print(f'斷詞完成。有標籤日期：{len(labeled_dates)} 天')
print(f'使用 TOKENIZE_METHOD={TOKENIZE_METHOD}，NGRAM_RANGE={NGRAM_RANGE}')

In [ ]:
# ── 向量化函式（與 Step 3 保持一致：固定 vocabulary + NGRAM_RANGE）──
from sklearn.feature_extraction.text import TfidfVectorizer as _TF

# 固定詞彙表（與 Step 3 對齊，含 N-gram 詞組）
_feature_vocab = {w: i for i, w in enumerate(feature_names)}

def vectorize_days(text_list: list):
    """
    將多個每日合併文本轉為 TF-IDF 向量矩陣。
    - vocabulary 固定為 Step 3 的 feature_names
    - ngram_range 與 Step 3 一致（NGRAM_RANGE）
    - use_idf=False：回測窗口樣本少，用 raw TF 避免 IDF 偏差；且 MultinomialNB 需非負值
    """
    vec = _TF(vocabulary=_feature_vocab, use_idf=False, lowercase=False,
              token_pattern=r'\S+', ngram_range=NGRAM_RANGE)
    return vec.fit_transform(text_list)   # fit_transform 僅計算 TF（vocabulary 固定不變）

In [ ]:
# ── 移動回測主迴圈（每天一筆，使用每日合併文本）──
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix

results_bt = []   # (date, pred_dir, actual_dir)

for test_date in labeled_dates:
    # ── 訓練窗口：前 WINDOW_DAYS 個日曆天，只取有標籤的日期 ──
    window_start = test_date - timedelta(days=WINDOW_DAYS)
    train_dates  = [d for d in labeled_dates if window_start <= d < test_date]
    train_texts  = [day_texts[d] for d in train_dates]
    train_y      = [day_labels[d] for d in train_dates]

    # ── 門檻 1：當日文章篇數不足 ──
    if day_counts.get(test_date, 0) < MIN_ARTICLES:
        continue

    # ── 門檻 2：訓練集需同時含看漲/看跌 ──
    if len(train_texts) < 2 or len(set(train_y)) < 2:
        continue

    # ── 向量化 ──
    X_tr = vectorize_days(train_texts)
    X_te = vectorize_days([day_texts[test_date]])

    # ── 訓練與預測 ──
    model = MultinomialNB()
    model.fit(X_tr, train_y)
    pred_dir   = int(model.predict(X_te)[0])
    actual_dir = day_labels[test_date]

    results_bt.append((test_date, pred_dir, actual_dir))

print(f'回測完成，出手 {len(results_bt)} 次（共 {len(labeled_dates)} 個有標籤日）')

In [ ]:
# ── 回測結果彙整 ──
n_total = len(all_dates)
n_trade = len(results_bt)
trade_rate = n_trade / n_total * 100 if n_total else 0

print('=' * 40)
print('移動回測結果')
print('=' * 40)
print(f'總文章天數：{n_total} 天')
print(f'出手天數：  {n_trade} 天（出手率：{trade_rate:.1f}%）')

if n_trade > 0:
    y_actual    = [r[2] for r in results_bt]
    y_predicted = [r[1] for r in results_bt]
    cm  = confusion_matrix(y_actual, y_predicted, labels=[1, -1])
    acc = accuracy_score(y_actual, y_predicted)

    print('\nConfusion Matrix：')
    print(f'{"":10s}預測漲  預測跌')
    print(f'真實漲    {cm[0][0]:5d}  {cm[0][1]:5d}')
    print(f'真實跌    {cm[1][0]:5d}  {cm[1][1]:5d}')
    print(f'\n總準確率：{acc * 100:.1f}%')
else:
    print('（沒有符合出手條件的日期）')

In [ ]:
# ── 逐日預測明細（前 20 筆）──
print(f'{"日期":12s}  預測  實際  結果')
print('-' * 35)
for date, pred, actual in results_bt[:20]:
    pred_s   = '漲' if pred   ==  1 else '跌'
    actual_s = '漲' if actual ==  1 else '跌'
    ok       = '✓' if pred == actual else '✗'
    print(f'{str(date):12s}  {pred_s}     {actual_s}     {ok}')